# Cinema Sales & Customer Analytics

Учебный проект по продажам сети из трёх кинотеатров. Задача — сравнить билетную выручку и использование вместимости, изучить каналы продаж и клиентские группы.

Период сеансов: 1 марта — 31 июля 2024 года. Полная локальная сборка выполняется из шести CSV. Подключение к учебной базе не требуется. [Интерактивный дашборд](https://datalens.ru/5e5xahyokzwip-cinema-sales-customer-analytics?tab=L1).


## Данные и метод

Одна строка продажи считается одним билетом. Сеанс определяется датой, частью дня, кинотеатром и залом. Сеансовая витрина начинается с полного расписания. Клиентская частота — число уникальных сеансов с покупкой, не число строк платежей.

Для бара явно выбрана политика `exact`: удаление полных совпадений по четырём столбцам, как в исходном ноутбуке. Она документируется как правило проекта; без идентификаторов позиций природу совпадений независимо установить нельзя. Из `clients` используются только ID и скидка.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys
import pandas as pd

base = Path.cwd().resolve()
if not (base / 'src' / 'build_from_raw.py').is_file():
    base = base.parent
if not (base / 'src' / 'build_from_raw.py').is_file():
    raise FileNotFoundError('Откройте ноутбук из корня репозитория или папки notebooks')
sys.path.insert(0, str(base / 'src'))
from build_from_raw import build

with TemporaryDirectory(prefix='.notebook-rebuild-', dir=base) as temp_dir:
    target = Path(temp_dir)
    assert target.resolve().is_relative_to(base.resolve())
    report = build(base / 'data' / 'raw', target, 'Europe/Moscow', 'exact')
    sessions = pd.read_csv(target / 'session_metrics.csv')
    customers = pd.read_csv(target / 'client_metrics.csv')
    monthly = pd.read_csv(target / 'monthly_sessions.csv')
    cinemas = pd.read_csv(target / 'cinema_comparison.csv')

print('Сборка из шести CSV завершена; ключи, JOIN и контрольные итоги проверены.')
print('Политика очистки бара:', report['bar']['duplicate_policy'])

Сборка из шести CSV завершена; ключи, JOIN и контрольные итоги проверены.
Политика очистки бара: exact


## Контрольные итоги

In [2]:
checks = pd.DataFrame({
    'Показатель': ['Сеансы расписания', 'Сеансы без продаж', 'Билеты', 'Покупатели', 'Билетная выручка, ₽', 'Бар после очистки, ₽'],
    'Значение': [report['schedule_sessions'], report['sessions_without_sales'], report['ticket_rows'], report['ticket_customers'], report['ticket_revenue'], report['bar']['bar_revenue_after_cleaning']]
})
display(checks)
print('Билетная выручка и количество билетов согласованы между продажами, сеансами и клиентами.')

,Показатель,Значение
0,Сеансы расписания,1963.0
1,Сеансы без продаж,0.0
2,Билеты,9643.0
3,Покупатели,9573.0
4,"Билетная выручка, ₽",2876035.0
5,"Бар после очистки, ₽",3393237.0


Билетная выручка и количество билетов согласованы между продажами, сеансами и клиентами.


## Очистка бара

Воспроизводится правило исходного учебного проекта: удаление полных совпадений client_id, pay_at, name, price. Без идентификатора позиции/заказа нельзя независимо установить, являются ли совпадения техническими дублями или несколькими одинаковыми товарами. Поэтому выбранная политика задаётся явно; ниже показано её влияние на строки и выручку.


In [3]:
bar = report['bar']
cleaning = pd.DataFrame({
    'Сценарий': ['Исходные строки', 'После удаления полных совпадений', 'Удалено'],
    'Строки': [bar['source_item_rows'], bar['retained_item_rows'], bar['removed_item_rows']],
    'Сумма, ₽': [bar['all_raw_bar_revenue'], bar['bar_revenue_after_cleaning'], bar['removed_bar_revenue']]
})
display(cleaning)

,Сценарий,Строки,"Сумма, ₽"
0,Исходные строки,16875,3786431.0
1,После удаления полных совпадений,14941,3393237.0
2,Удалено,1934,393194.0


## Динамика и сравнение кинотеатров

Месяц сеанса определяется по session_date, а не по дате одной из оплат. Календарь оплат рассчитывается отдельно. Заполняемость — отношение всех проданных мест к сумме вместимости сеансов.


In [4]:
display(monthly)
display(cinemas[['cinema_id', 'scheduled_sessions', 'ticket_revenue', 'occupancy_pct', 'revenue_per_available_seat', 'revenue_per_scheduled_session', 'online_share_pct']])

,month_start,scheduled_sessions,tickets,ticket_revenue,available_seats,occupancy_pct
0,2024-03-01 00:00:00,462,2461,756254.0,10248,24.014442
1,2024-04-01 00:00:00,431,2041,667383.0,9764,20.903318
2,2024-05-01 00:00:00,427,2394,769392.5,9740,24.579055
3,2024-06-01 00:00:00,405,2210,550048.5,8948,24.698257
4,2024-07-01 00:00:00,238,537,132957.0,5232,10.263761


,cinema_id,scheduled_sessions,ticket_revenue,occupancy_pct,revenue_per_available_seat,revenue_per_scheduled_session,online_share_pct
0,AC1,499,713419.0,19.654140,59.313186,1429.697395,65.270728
1,AC2,735,1195670.0,23.339548,69.661501,1626.761905,67.099351
2,AC3,729,966946.0,22.204885,65.600136,1326.400549,66.911091


## Клиентские группы

In [5]:
segments = customers.groupby('client_type', as_index=False).agg(
    clients=('client_id', 'size'),
    sessions_with_purchase=('sessions_visited', 'sum'),
    revenue_per_client=('total_revenue_with_bar', 'mean')
)
display(segments)
print('Выручка на клиента за период:', round(customers.total_revenue_with_bar.mean(), 2), '₽')

,client_type,clients,sessions_with_purchase,revenue_per_client
0,1 сеанс,9504,9504,650.029724
1,2–3 сеанса,69,139,1324.485507


Выручка на клиента за период: 654.89 ₽


## Заполняемость по времени суток и составляющие июльского спада


In [6]:
dayparts = sessions.groupby('part_of_day', as_index=False).agg(
    sessions=('session_key', 'size'),
    tickets=('session_tickets_sold', 'sum'),
    available_seats=('count_seats', 'sum'),
    ticket_revenue=('session_revenue', 'sum'),
)
dayparts['occupancy_pct'] = 100 * dayparts.tickets / dayparts.available_seats
display(dayparts)

may = monthly.loc[monthly.month_start.str.startswith('2024-05')].iloc[0]
july = monthly.loc[monthly.month_start.str.startswith('2024-07')].iloc[0]
print('Изменение билетной выручки, июль к маю:', round(100 * (july.ticket_revenue / may.ticket_revenue - 1), 2), '%')
print('Изменение числа сеансов, июль к маю:', round(100 * (july.scheduled_sessions / may.scheduled_sessions - 1), 2), '%')
print('Доля бара в совокупной выручке:', round(100 * bar['bar_revenue_after_cleaning'] / (report['ticket_revenue'] + bar['bar_revenue_after_cleaning']), 2), '%')


,part_of_day,sessions,tickets,available_seats,ticket_revenue,occupancy_pct
0,вечерний,668,5798,15480,1686443.0,37.454780
1,дневной,774,2713,17064,841097.0,15.898969
2,утренний,521,1132,11388,348495.0,9.940288


Изменение билетной выручки, июль к маю: -82.72 %
Изменение числа сеансов, июль к маю: -44.26 %
Доля бара в совокупной выручке: 54.12 %


## Выводы для управления

1. В июле билетная выручка снизилась на 83% относительно мая: с 769,4 до 133,0 тыс. ₽. Одновременно стало на 44% меньше сеансов, заполняемость снизилась с 24,6% до 10,3%, средняя цена проданного билета — с 321 до 248 ₽.
2. Вечерние сеансы заполняются лучше: 37,5% против 15,9% днём и 9,9% утром. Утренние и дневные показы — приоритет для проверки расписания, репертуара и предложений зрителям.
3. AC2 даёт 41,6% билетной выручки сети и лидирует по выручке на доступное место: 69,7 ₽ против 59,3 ₽ у AC1 и 65,6 ₽ у AC3.
4. Бар даёт 54,1% совокупной выручки: 3,39 млн ₽ из 6,27 млн ₽ при объявленном правиле очистки.

Различия описывают данные, но не доказывают причины. Для проверки продвижения утренних и дневных сеансов подготовлен [дизайн A/B-теста](../docs/AB_TEST_DESIGN.md). Сам эксперимент не проводился.


## Ограничения

- Правило удаления полных совпадений бара задано явно; техническая природа совпадений не установлена независимо.
- Бар не имеет ключа кинотеатра/сеанса и идентификатора чека; его не распределяют по кинотеатрам и отдельным визитам.
- История цен и скидок отсутствует, используются предоставленные справочники.
- Покупка не подтверждает фактическое посещение. История до окна наблюдения неизвестна.
- CSV не содержит смещения часового пояса: сборка предполагает московское время. Анализ времени суток в проекте использует поле part_of_day расписания, а не часы оплаты.
